# ItemKNN Baseline trong bài toán gợi ý phim với phản hồi ẩn

## Tóm tắt (Abstract)
Notebook này triển khai mô hình Item-based K-Nearest Neighbors (ItemKNN) như một baseline lọc cộng tác (collaborative filtering) cổ điển. Mô hình xây dựng ma trận tương tác nhị phân $R$ từ phản hồi ẩn, tính độ tương đồng cosine giữa các item và xếp hạng item ứng viên dựa trên các item lân cận đã được người dùng tương tác trong tập huấn luyện.

## 1. Giới thiệu (Introduction)
ItemKNN là mốc so sánh quan trọng vì mô hình chỉ dựa trên cấu trúc đồng tương tác giữa item, không sử dụng metadata văn bản, hồ sơ người dùng hoặc embedding tri thức. Do đó, kết quả của notebook này giúp tách riêng đóng góp của tín hiệu lọc cộng tác khỏi các thành phần biểu diễn nội dung và đồ thị tri thức trong các mô hình LightFM/SeRel-LightFM.

| Thành phần | Mô tả |
|---|---|
| Thiết lập chia dữ liệu | Leave-One-Out theo từng người dùng |
| Ma trận tương tác | Nhị phân hóa theo $r_{u,i} \geq \tau$ |
| Độ tương đồng item | Cosine similarity trên vector người dùng-item |
| Chiến lược gợi ý | Tổng hợp điểm tương đồng từ $K$ item lân cận |


## 2. Phương pháp nghiên cứu (Methodology)

### 2.1 Thiết lập môi trường và cấu hình
Cell cấu hình khai báo nguồn dữ liệu, ngưỡng phản hồi ẩn $\tau$, số lượng rating tối thiểu trên mỗi người dùng, danh sách cutoff Top-$K$ và các giá trị số láng giềng ứng viên của ItemKNN.


In [1]:
# [Giải thích cell]
# - Mục đích: nạp thư viện và khai báo toàn bộ cấu hình thực nghiệm của notebook.
# - Đầu vào chính: URL dữ liệu, thư mục checkpoint, ngưỡng phản hồi ẩn và siêu tham số mô hình.
# - Kết quả tạo ra: các hằng số cấu hình được các cell phía sau sử dụng thống nhất.
# - Lưu ý: cấu hình thuộc biến thể ItemKNN Baseline, do đó cần giữ cố định khi so sánh với notebook khác.

import os, warnings
from collections import Counter

import pandas as pd
import numpy as np
from scipy.sparse import csr_matrix
from scipy.sparse.linalg import norm as sparse_norm

warnings.filterwarnings("ignore", category=FutureWarning)

# ── Nguồn dữ liệu ───────────────────────────────────────────────────────────────
MOVIES_URL  = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/movies.csv"
RATINGS_URL = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/ratings.csv"
USERS_URL   = "https://raw.githubusercontent.com/lynchblue/movie-rating-dataset/main/data/user_profiles.csv"

# ── Cấu hình thực nghiệm ──────────────────────────────────────────────────────────────
POSITIVE_THRESHOLD = 7       # Binary implicit: rate >= này → positive
MIN_RATINGS        = 10      # Lọc user có ít hơn N ratings
K_NEIGHBOURS       = 50      # Giá trị mặc định; sẽ được chọn lại bằng validation
ITEM_K_LIST        = [5, 10, 20, 50]  # Candidate neighbour sizes cho ItemKNN
K_VALUES           = (5, 10, 20, 50)  # Evaluation cutoffs

print(f"[ItemKNN] Cấu hình: threshold={POSITIVE_THRESHOLD}, default_K_neighbours={K_NEIGHBOURS}")
print(f"[ItemKNN] ITEM_K_LIST = {ITEM_K_LIST}")
print(f"[ItemKNN] Split : Leave-One-Out (LOO) per user")
print(f"[ItemKNN] Similarity: Cosine (binary implicit matrix)")
print(f"[ItemKNN] NO content features, NO side information")

[ItemKNN] Config: threshold=7, default_K_neighbours=50
[ItemKNN] ITEM_K_LIST = [5, 10, 20, 50]
[ItemKNN] Split : Leave-One-Out (LOO) per user
[ItemKNN] Similarity: Cosine (binary implicit matrix)
[ItemKNN] NO content features, NO side information


### 2.2 Các hàm hỗ trợ
Các hàm hỗ trợ chuẩn hóa văn bản và xử lý token được giữ thống nhất với các notebook còn lại. Điều này bảo đảm khác biệt kết quả đến từ mô hình gợi ý, không đến từ sai khác tiền xử lý.


In [2]:
# [Giải thích cell]
# - Mục đích: định nghĩa các hàm tiền xử lý văn bản và token dùng lại trong notebook.
# - Đầu vào chính: các cột metadata dạng chuỗi, thường chứa token phân tách bằng dấu `|`.
# - Kết quả tạo ra: chuỗi đã làm sạch, danh sách token và tập token đủ phổ biến để làm đặc trưng.
# - Lưu ý: các hàm này chỉ chuẩn hóa dữ liệu; không học tham số từ validation hoặc test.

def cleanup_column(series):
    """Fix encoding artefacts in pipe-separated string Series."""
    replacements = {
        '| ': '|', '\xc3\xa9': 'é', '\xc3\x81': 'Á', '\xc3\x93': 'Ó',
        '\xc3\xa1': 'á', '\xc3\xb3': 'ó', '\xc3\xb1': 'ñ', '\xc3\xad': 'í',
        '\xc3\xba': 'ú', 'Ã©': 'é', 'Ã¡': 'á', 'Ã³': 'ó', 'Ã±': 'ñ',
        'Ãº': 'ú',
    }
    col = series.copy()
    for bad, good in replacements.items():
        col = col.str.replace(bad, good, regex=False)
    return col

print("Helpers defined: cleanup_column")

Helpers defined: cleanup_column


### 2.3 Nạp dữ liệu
Các bảng phim, rating và hồ sơ người dùng được nạp và chuẩn hóa kiểu dữ liệu thời gian. Dữ liệu rating là nguồn tín hiệu chính để xây dựng ma trận tương tác nhị phân.


In [3]:
# [Giải thích cell]
# - Mục đích: nạp ba bảng dữ liệu gốc gồm rating, hồ sơ người dùng và metadata phim.
# - Đầu vào chính: các đường dẫn `RATINGS_URL`, `USERS_URL` và `MOVIES_URL`.
# - Kết quả tạo ra: `ratings_df`, `users_df`, `movies_df` và cột thời gian đã được chuẩn hóa.
# - Lưu ý: kiểm tra kích thước bảng sau khi nạp giúp phát hiện sớm lỗi đọc dữ liệu.

ratings_df = pd.read_csv(RATINGS_URL)
users_df   = pd.read_csv(USERS_URL)
movies_df  = pd.read_csv(MOVIES_URL)

ratings_df["date"] = pd.to_datetime(ratings_df["date"])

print(f"Users   : {users_df['id'].nunique()}")
print(f"Movies  : {movies_df['id'].nunique()}")
print(f"Ratings : {len(ratings_df):,}")
print(f"Rating range: {ratings_df['rate'].min()} – {ratings_df['rate'].max()}")
print(f"Date range  : {ratings_df['date'].min().date()} – {ratings_df['date'].max().date()}")

Users   : 482
Movies  : 78628
Ratings : 1,172,038
Rating range: 1 – 10
Date range  : 2002-08-14 – 2021-04-01


### 2.4 Lọc dữ liệu hợp lệ
Quy trình lọc loại bỏ rating không có hồ sơ người dùng, người dùng có quá ít rating và movie không tồn tại trong bảng metadata. Bước này bảo đảm ma trận tương tác có chỉ mục nhất quán.


In [4]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

# 2.1 Lọc orphan users (có rating nhưng KHÔNG có profile)
users_in_profile = set(users_df["id"].unique())
ratings_df = ratings_df[ratings_df["id"].isin(users_in_profile)].copy()
print(f"Sau lọc orphan users    : {len(ratings_df):,} ratings")

# 2.2 Lọc user có ít hơn MIN_RATINGS ratings
rating_counts = ratings_df.groupby("id")["movie_id"].count()
valid_users   = rating_counts[rating_counts >= MIN_RATINGS].index
ratings_df    = ratings_df[ratings_df["id"].isin(valid_users)].copy()
users_df      = users_df[users_df["id"].isin(valid_users)].copy()
print(f"Sau lọc min {MIN_RATINGS} ratings : {len(ratings_df):,} ratings, {ratings_df['id'].nunique()} users")

# 2.3 Lọc movie không tồn tại trong movies.csv
movies_in_table = set(movies_df["id"].unique())
ratings_df = ratings_df[ratings_df["movie_id"].isin(movies_in_table)].copy()
print(f"Sau lọc missing movies  : {len(ratings_df):,} ratings")

Sau lọc orphan users    : 963,266 ratings
Sau lọc min 10 ratings : 963,231 ratings, 474 users
Sau lọc missing movies  : 963,228 ratings


### 2.5 Chuẩn hóa metadata văn bản
Các cột văn bản được chuẩn hóa để phục vụ hiển thị và phân tích phụ trợ. ItemKNN không dùng metadata trong quá trình tính điểm, do đó phần này không tạo rò rỉ dữ liệu vào mô hình.


In [5]:
# [Giải thích cell]
# - Mục đích: thực hiện một bước xử lý trong pipeline thực nghiệm của notebook.
# - Đầu vào chính: các biến đã được tạo ở những cell trước.
# - Kết quả tạo ra: biến trung gian hoặc bảng kết quả dùng cho các bước tiếp theo.
# - Lưu ý: comment này được bổ sung để giúp người đọc theo dõi luồng xử lý mà không cần chạy lại code.

for col in ["country_name", "genres", "topics", "directors", "actors",
            "script", "producer", "music", "photography"]:
    movies_df[col] = cleanup_column(movies_df[col])

print("Text cleanup hoàn tất.")

Text cleanup hoàn tất.


### 2.6 Chia dữ liệu Leave-One-Out và áp dụng threshold
Quy trình dữ liệu được tách thành hai giai đoạn rõ ràng:

1. **Chia Leave-One-Out trên rating gốc:** `ratings_df` sau khi lọc hợp lệ vẫn giữ đầy đủ giá trị rating. Với mỗi người dùng, rating được sắp xếp theo thời gian; rating mới nhất được đưa vào `test_df`, rating liền trước được đưa vào `valid_df`, và toàn bộ rating còn lại được đưa vào `train_df`.
2. **Áp dụng threshold sau khi đã chia:** các split `train_df`, `valid_df` và `test_df` vẫn là dữ liệu rating gốc. Khi xây dựng tương tác phản hồi ẩn, chỉ các dòng thỏa $r_{u,i} \geq \tau$ mới được chuyển thành tương tác dương.

Điều này có nghĩa là threshold không được dùng để quyết định dòng nào thuộc train, validation hay test. Threshold chỉ quyết định dòng nào trở thành tương tác dương khi huấn luyện hoặc đánh giá. Cách làm này giữ đúng thứ tự thời gian của hành vi người dùng và tránh làm thay đổi tương tác mới nhất trước khi chia dữ liệu.


In [6]:
# [Giải thích cell]
# - Mục đích: chia dữ liệu Leave-One-Out trên rating gốc, trước khi áp dụng threshold phản hồi ẩn.
# - Cách làm: sắp xếp rating theo thời gian cho từng user; lấy rating mới nhất làm test, rating liền trước làm validation, phần còn lại làm train.
# - Kết quả tạo ra: `train_df`, `valid_df`, `test_df`; các bảng này vẫn giữ rating gốc, gồm cả rating cao và thấp.
# - Lưu ý: threshold `rate >= POSITIVE_THRESHOLD` chỉ được áp dụng ở các bước xây dựng interaction/evaluation sau đó.

ratings_df = ratings_df.sort_values(["id", "date"]).reset_index(drop=True)

train_idx = []
valid_idx = []
test_idx  = []

for uid, group in ratings_df.groupby("id", sort=False):
    idx = group.index.tolist()
    if len(idx) < 3:
        train_idx.extend(idx)
        continue
    test_idx.append(idx[-1])
    valid_idx.append(idx[-2])
    train_idx.extend(idx[:-2])

train_df = ratings_df.loc[train_idx].copy().reset_index(drop=True)
valid_df = ratings_df.loc[valid_idx].copy().reset_index(drop=True)
test_df  = ratings_df.loc[test_idx].copy().reset_index(drop=True)

# Tạo các tập held-out dương dùng cho đánh giá phản hồi ẩn
train_positive_df = train_df[train_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)
valid_positive_df = valid_df[valid_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)
test_positive_df  = test_df[test_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)

print(f"Train : {len(train_df):>7,} raw ratings  |  {train_df['id'].nunique()} users")
print(f"Valid : {len(valid_df):>7,} raw ratings  |  {valid_df['id'].nunique()} users  (1 rating/user)")
print(f"Test  : {len(test_df):>7,} raw ratings  |  {test_df['id'].nunique()} users  (1 rating/user)")
print()
print(f"Train positives (rate >= {POSITIVE_THRESHOLD}) : {len(train_positive_df):>7,} interactions | {train_positive_df['id'].nunique()} users")
print(f"Valid positives (rate >= {POSITIVE_THRESHOLD}) : {len(valid_positive_df):>7,} interactions | {valid_positive_df['id'].nunique()} evaluable users")
print(f"Test  positives (rate >= {POSITIVE_THRESHOLD}) : {len(test_positive_df):>7,} interactions | {test_positive_df['id'].nunique()} evaluable users")
print()
print(f"Date range train: {train_df['date'].min().date()} – {train_df['date'].max().date()}")
print(f"Date range valid: {valid_df['date'].min().date()} – {valid_df['date'].max().date()}")
print(f"Date range test : {test_df['date'].min().date()} – {test_df['date'].max().date()}")


Train : 962,280 raw ratings  |  474 users
Valid :     474 raw ratings  |  474 users  (1 rating/user)
Test  :     474 raw ratings  |  474 users  (1 rating/user)

Train positives (rate >= 7) : 412,438 interactions | 473 users
Valid positives (rate >= 7) :     240 interactions | 240 evaluable users
Test  positives (rate >= 7) :     262 interactions | 262 evaluable users

Date range train: 2002-08-14 – 2021-03-31
Date range valid: 2011-12-06 – 2021-04-01
Date range test : 2011-12-08 – 2021-04-01


### 2.7 Phân loại item warm/cold
Item warm là item đã xuất hiện trong train, trong khi item cold chưa có vector tương tác trong ma trận huấn luyện. Phân tích warm/cold đặc biệt quan trọng với ItemKNN vì mô hình cần lịch sử đồng tương tác để tính độ tương đồng item.


In [7]:
# [Giải thích cell]
# - Mục đích: tách item trong tập kiểm thử thành warm và cold theo sự xuất hiện trong train.
# - Đầu vào chính: `train_df` và `test_df`.
# - Kết quả tạo ra: `test_warm_df`, `test_cold_df` cùng các phiên bản positive nếu có.
# - Lưu ý: phân tách này giúp đánh giá riêng khả năng xử lý item cold-start.

movies_in_train = set(train_df["movie_id"].unique())

# Phân loại warm/cold trên ứng viên test thô
test_warm_mask = test_df["movie_id"].isin(movies_in_train)
test_cold_mask = ~test_warm_mask

test_warm_df = test_df[test_warm_mask].copy().reset_index(drop=True)
test_cold_df = test_df[test_cold_mask].copy().reset_index(drop=True)

# Phân loại warm/cold sau ngưỡng phản hồi ẩn; đây là ground truth dương thực tế
valid_positive_df = valid_df[valid_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)
test_positive_df  = test_df[test_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)
test_warm_positive_df = test_warm_df[test_warm_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)
test_cold_positive_df = test_cold_df[test_cold_df["rate"] >= POSITIVE_THRESHOLD].copy().reset_index(drop=True)

print(f"Unique items in train : {len(movies_in_train):,}")
print(f"Unique items in valid : {valid_df['movie_id'].nunique():,}")
print(f"Unique items in test  : {test_df['movie_id'].nunique():,}")
print()
print("Raw held-out test candidates:")
print(f"TEST total : {len(test_df):>6,} ratings")
print(f"TEST warm  : {len(test_warm_df):>6,} ratings  "
      f"({len(test_warm_df)/len(test_df)*100:.1f}%)  "
      f"— {test_warm_df['movie_id'].nunique():,} unique movies")
print(f"TEST cold  : {len(test_cold_df):>6,} ratings  "
      f"({len(test_cold_df)/len(test_df)*100:.1f}%)  "
      f"— {test_cold_df['movie_id'].nunique():,} unique movies")
print()
print(f"Positive held-out targets after threshold rate >= {POSITIVE_THRESHOLD}:")
print(f"VALID positive users : {valid_positive_df['id'].nunique():>6,}")
print(f"TEST positive users  : {test_positive_df['id'].nunique():>6,}")
print(f"TEST_WARM positive   : {test_warm_positive_df['id'].nunique():>6,}")
print(f"TEST_COLD positive   : {test_cold_positive_df['id'].nunique():>6,}")
print()
print("⚠ ItemKNN không có vector cho cold items → TEST_COLD metrics thường bằng 0.0")


Unique items in train : 75,115
Unique items in valid : 398
Unique items in test  : 407

Raw held-out test candidates:
TEST total :    474 ratings
TEST warm  :    446 ratings  (94.1%)  — 379 unique movies
TEST cold  :     28 ratings  (5.9%)  — 28 unique movies

Positive held-out targets after threshold rate >= 7:
VALID positive users :    240
TEST positive users  :    262
TEST_WARM positive   :    253
TEST_COLD positive   :      9

⚠ ItemKNN không có vector cho cold items → TEST_COLD metrics thường bằng 0.0


### 2.8 Xây dựng ánh xạ chỉ mục
Ánh xạ `user_id` và `movie_id` sang chỉ mục ma trận được tạo từ tập huấn luyện. Đây là bước trung gian cần thiết để biểu diễn dữ liệu dưới dạng sparse matrix.


In [8]:
# [Giải thích cell]
# - Mục đích: tạo ánh xạ giữa mã user/movie gốc và chỉ số hàng/cột trong ma trận.
# - Đầu vào chính: tập train đã lọc.
# - Kết quả tạo ra: `user2idx`, `movie2idx`, `idx2user` và `idx2movie`.
# - Lưu ý: chỉ mục ổn định là điều kiện cần để xây dựng sparse matrix chính xác.

# Tập hợp toàn bộ users và items trong tập train
all_users  = sorted(train_df["id"].unique().tolist())
all_movies = sorted(train_df["movie_id"].unique().tolist())

user2idx  = {u: i for i, u in enumerate(all_users)}
movie2idx = {m: i for i, m in enumerate(all_movies)}
idx2movie = {i: m for m, i in movie2idx.items()}
idx2user  = {i: u for u, i in user2idx.items()}

n_users  = len(all_users)
n_movies = len(all_movies)

print(f"Users trong train  : {n_users:,}")
print(f"Movies trong train : {n_movies:,}")

Users trong train  : 474
Movies trong train : 75,115


### 2.9 Xây dựng ma trận tương tác nhị phân
Ma trận $R \in \{0,1\}^{|U| \times |I|}$ được xây dựng từ các tương tác dương trong train:

$$
R_{u,i} = \mathbb{1}(r_{u,i} \geq \tau).
$$

Rating gốc không được dùng làm trọng số để giữ đúng thiết lập phản hồi ẩn.


In [9]:
# [Giải thích cell]
# - Mục đích: xây dựng ma trận user-item nhị phân cho ItemKNN từ train sau threshold.
# - Đầu vào train model: các dòng trong `train_df` có `rate >= POSITIVE_THRESHOLD`.
# - Kết quả tạo ra: sparse matrix `R`, dùng để tính độ tương đồng item.
# - Lưu ý: rating thấp hơn ngưỡng không đóng góp vào vector tương tác dương.

# Chỉ lấy positive interactions (rate >= ngưỡng)
train_positive = train_df[train_df["rate"] >= POSITIVE_THRESHOLD].copy()

# Lọc các user/movie có trong mapping
train_positive = train_positive[
    train_positive["id"].isin(user2idx) &
    train_positive["movie_id"].isin(movie2idx)
]

row_indices = train_positive["id"].map(user2idx).values
col_indices = train_positive["movie_id"].map(movie2idx).values
data        = np.ones(len(train_positive), dtype=np.float32)

# Ma trận user-item (binary)
R = csr_matrix(
    (data, (row_indices, col_indices)),
    shape=(n_users, n_movies),
    dtype=np.float32,
)

print(f"User-Item matrix shape : {R.shape}")
print(f"Positive interactions  : {R.nnz:,}")
print(f"Matrix density         : {R.nnz / (n_users * n_movies) * 100:.4f}%")
print(f"Threshold              : rate >= {POSITIVE_THRESHOLD}")

User-Item matrix shape : (474, 75115)
Positive interactions  : 412,438
Matrix density         : 1.1584%
Threshold              : rate >= 7


### 2.10 Tính độ tương đồng cosine giữa item
Độ tương đồng giữa hai item $i$ và $j$ được tính trên các cột của ma trận $R$:

$$
\operatorname{sim}(i,j) = \frac{\mathbf{r}_i^\top \mathbf{r}_j}{\|\mathbf{r}_i\|_2 \|\mathbf{r}_j\|_2}.
$$

Đường chéo được loại bỏ để item không tự đóng góp vào điểm của chính nó.


In [10]:
# [Giải thích cell]
# - Mục đích: chuẩn hóa vector item để tính cosine similarity hiệu quả.
# - Đầu vào chính: ma trận tương tác `R`.
# - Kết quả tạo ra: `R_T_norm`, ma trận item-user đã chuẩn hóa L2.
# - Lưu ý: sau chuẩn hóa, tích vô hướng giữa hai vector item chính là cosine similarity.

from sklearn.preprocessing import normalize

# Chuẩn hóa L2 theo cột (mỗi item là 1 cột trong R)
# Chuyển sang item-user: shape (n_movies, n_users)
R_T = R.T.tocsr()  # shape: (n_movies, n_users)

# Normalize L2 mỗi item vector (mỗi hàng của R_T)
R_T_norm = normalize(R_T, norm='l2', axis=1)  # shape: (n_movies, n_users)

print(f"Item-User matrix (R_T) shape : {R_T.shape}")
print(f"Normalized R_T shape         : {R_T_norm.shape}")
print(f"[ItemKNN] Similarity sẽ được tính on-the-fly per user tại inference")
print(f"[ItemKNN] K_NEIGHBOURS = {K_NEIGHBOURS}")

Item-User matrix (R_T) shape : (75115, 474)
Normalized R_T shape         : (75115, 474)
[ItemKNN] Similarity sẽ được tính on-the-fly per user tại inference
[ItemKNN] K_NEIGHBOURS = 50


### 2.11 Hàm chấm điểm ItemKNN
Với người dùng $u$ và item ứng viên $j$, điểm gợi ý được tính bằng tổng độ tương đồng với các item dương mà người dùng đã tương tác:

$$
\operatorname{score}(u,j) = \sum_{i \in N(j,K) \cap I_u^+} \operatorname{sim}(i,j).
$$

Trong đó $N(j,K)$ là tập $K$ láng giềng gần nhất của item $j$ và $I_u^+$ là tập item dương trong train của người dùng $u$.


In [11]:
# [Giải thích cell]
# - Mục đích: tính điểm ItemKNN cho toàn bộ item ứng viên của một user.
# - Đầu vào chính: vector tương tác của user và ma trận item đã chuẩn hóa.
# - Kết quả tạo ra: mảng điểm gợi ý cho tất cả item trong catalogue train.
# - Lưu ý: chỉ các láng giềng gần nhất trong lịch sử user được giữ lại khi cộng điểm.

def get_user_scores_itemknn(user_idx, R, R_T_norm, k_neighbours=K_NEIGHBOURS):
    """
    Tính score itemKNN cho TẤT CẢ items trong catalogue với 1 user.

    Parameters
    ----------
    user_idx   : int — row index của user trong R
    R          : csr_matrix (n_users, n_movies) — binary interaction matrix
    R_T_norm   : csr_matrix (n_movies, n_users) — L2-normalized item-user matrix
    k_neighbours: int — số lân cận dùng để tính score

    Returns
    -------
    scores : np.ndarray shape (n_movies,) — score cho từng item
    """
    n_movies = R_T_norm.shape[0]

    # Lấy tập items user đã tương tác (positive) trong train
    user_row   = R[user_idx]              # shape (1, n_movies)
    user_items = user_row.indices         # index của items đã tương tác

    if len(user_items) == 0:
        return np.zeros(n_movies, dtype=np.float32)

    # Lấy item vectors của các items user đã tương tác
    # shape: (n_interacted, n_users)
    interacted_vecs = R_T_norm[user_items]  # sparse

    # Tính similarity giữa MỌI items trong catalogue và các items user đã tương tác
    # sim_matrix shape: (n_movies, n_interacted)
    # = R_T_norm (n_movies, n_users) @ interacted_vecs.T (n_users, n_interacted)
    sim_matrix = R_T_norm.dot(interacted_vecs.T)  # (n_movies, n_interacted)

    # Chuyển về dense
    sim_dense = np.asarray(sim_matrix.todense(), dtype=np.float32)  # (n_movies, n_interacted)

    # Với mỗi candidate item, chỉ giữ Top-K neighbors trong số các items đã tương tác
    # Nếu n_interacted <= k_neighbours: giữ tất cả
    if sim_dense.shape[1] > k_neighbours:
        # Sắp xếp theo chiều ngang, lấy top-K similarity
        top_k_idx  = np.argpartition(-sim_dense, k_neighbours, axis=1)[:, :k_neighbours]
        rows       = np.arange(n_movies)[:, None]
        sim_top_k  = sim_dense[rows, top_k_idx]
        scores     = sim_top_k.sum(axis=1)
    else:
        scores = sim_dense.sum(axis=1)  # (n_movies,)

    return scores


print("[ItemKNN] Scoring function defined: get_user_scores_itemknn")
print(f"          Neighbourhood size K = {K_NEIGHBOURS}")

[ItemKNN] Scoring function defined: get_user_scores_itemknn
          Neighbourhood size K = 50


## 3. Thực nghiệm và Kết quả (Experiments & Results)

### 3.1 Các chỉ số đánh giá
Các chỉ số Precision@$K$, Recall@$K$, NDCG@$K$, HR@$K$ và MRR@$K$ được tính trên các item dương ở validation/test. Khi xếp hạng, toàn bộ item đã xuất hiện trong lịch sử train của người dùng được mask khỏi danh sách ứng viên.


In [12]:
# [Giải thích cell]
# - Mục đích: định nghĩa hàm đánh giá Top-K trên các held-out interactions dương.
# - Đầu vào chính: ma trận held-out sau threshold, `train_interactions` để mask lịch sử và mô hình sinh điểm.
# - Kết quả tạo ra: Precision@K, Recall@K, NDCG@K, HR@K và MRR@K.
# - Lưu ý: validation/test chỉ được dùng để đo hiệu năng; không cập nhật trọng số mô hình.

def evaluate_metrics_itemknn(
    eval_df,
    train_df,
    R,
    R_T_norm,
    user2idx,
    movie2idx,
    idx2movie,
    k_values=(5, 10, 20, 50),
    k_neighbours=K_NEIGHBOURS,
    threshold=POSITIVE_THRESHOLD,
):
    """
    Tính Precision@K, Recall@K, NDCG@K, HR@K, MRR@K cho nhiều K
    theo thiết lập implicit feedback.

    Quan trọng:
    - eval_df có thể là raw validation/test split.
    - Hàm này chỉ giữ held-out items có rate >= ngưỡng làm positive targets.
    - Users không có positive held-out item sau threshold sẽ bị bỏ qua.
    - Đây là thiết lập nhất quán với LightFM / SeRel-LightFM.

    Returns
    -------
    dict {K: {metric: float, n_users: int}}
    """
    if eval_df is None or len(eval_df) == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                    "hr": 0., "mrr": 0., "n_users": 0} for k in k_values}

    # Chỉ evaluate users có positive held-out items (rate >= ngưỡng)
    eval_pos = eval_df[eval_df["rate"] >= threshold].copy()
    if len(eval_pos) == 0:
        return {k: {"precision": 0., "recall": 0., "ndcg": 0.,
                    "hr": 0., "mrr": 0., "n_users": 0} for k in k_values}

    max_k  = max(k_values)
    accum  = {k: {"precision": [], "recall": [], "ndcg": [],
                  "hr": [], "mrr": []} for k in k_values}

    # Theo từng người dùng: tập movie_id trong train cần loại khỏi gợi ý.
    # Loại toàn bộ rating train, không chỉ rating dương, vì đây là các item người dùng đã tiếp xúc.
    train_seen = train_df.groupby("id")["movie_id"].apply(set).to_dict()

    # Theo từng người dùng: các item dương held-out làm ground truth
    # Với LOO, mỗi người dùng thường có một item dương sau khi áp ngưỡng.
    eval_by_user = eval_pos.groupby("id")["movie_id"].apply(set).to_dict()

    n_movies = R_T_norm.shape[0]

    for user_id, true_movie_ids in eval_by_user.items():
        if user_id not in user2idx:
            # User không có trong train → bỏ qua
            continue

        u_idx = user2idx[user_id]

        # Tính scores cho tất cả train-catalog items
        scores = get_user_scores_itemknn(u_idx, R, R_T_norm, k_neighbours)

        # Mask items đã thấy trong raw train history (set score = -inf)
        seen_ids = train_seen.get(user_id, set())
        for mid in seen_ids:
            if mid in movie2idx:
                scores[movie2idx[mid]] = -np.inf

        # Lấy top max_k
        top_k = min(max_k, n_movies)
        top_idx = np.argpartition(-scores, top_k - 1)[:top_k]
        top_idx = top_idx[np.argsort(-scores[top_idx])]

        # Chuyển index về movie_id
        top_movie_ids = [idx2movie[i] for i in top_idx]

        # Relevance vector
        relevance  = np.array([1.0 if m in true_movie_ids else 0.0
                               for m in top_movie_ids])
        n_relevant = len(true_movie_ids)

        for k in k_values:
            rel_k  = relevance[:k]
            n_hits = float(rel_k.sum())

            precision = n_hits / k
            recall    = n_hits / n_relevant if n_relevant > 0 else 0.0
            hr        = 1.0 if n_hits > 0 else 0.0

            discounts = 1.0 / np.log2(np.arange(2, k + 2))
            dcg  = float((rel_k * discounts).sum())
            idcg = float(discounts[:min(n_relevant, k)].sum())
            ndcg = dcg / idcg if idcg > 0 else 0.0

            mrr = 0.0
            for j in range(min(k, len(rel_k))):
                if rel_k[j] > 0:
                    mrr = 1.0 / (j + 1)
                    break

            accum[k]["precision"].append(precision)
            accum[k]["recall"].append(recall)
            accum[k]["ndcg"].append(ndcg)
            accum[k]["hr"].append(hr)
            accum[k]["mrr"].append(mrr)

    summary = {}
    for k in k_values:
        summary[k] = {m: float(np.mean(v)) if v else 0.0
                      for m, v in accum[k].items()}
        summary[k]["n_users"] = len(accum[k]["precision"])
    return summary


def print_metrics(metrics, label):
    n_users = list(metrics.values())[0].get("n_users", "?")
    print(f"\n=== {label} (n_users={n_users}) ===")
    print(f"{'K':>4} | {'Prec':>8} | {'Recall':>8} | {'NDCG':>8} | {'HR':>8} | {'MRR':>8}")
    print("-" * 58)
    for k, m in sorted(metrics.items()):
        print(f"{k:>4} | {m['precision']:>8.4f} | {m['recall']:>8.4f} | "
              f"{m['ndcg']:>8.4f} | {m['hr']:>8.4f} | {m['mrr']:>8.4f}")


print("Metrics defined: evaluate_metrics_itemknn, print_metrics")
print("Evaluation uses only positive held-out items: rate >= POSITIVE_THRESHOLD")


Metrics defined: evaluate_metrics_itemknn, print_metrics
Evaluation uses only positive held-out items: rate >= POSITIVE_THRESHOLD


### 3.2 Lựa chọn số láng giềng
Tập validation được sử dụng để lựa chọn `K_NEIGHBOURS`, tức số item lân cận tham gia vào công thức chấm điểm. Bảng 1 trình bày NDCG@10 theo từng giá trị ứng viên.


In [13]:
# [Giải thích cell]
# - Mục đích: chọn số láng giềng ItemKNN tốt nhất bằng tập validation.
# - Đầu vào chính: danh sách `ITEM_K_LIST` và hàm đánh giá ItemKNN.
# - Kết quả tạo ra: `K_NEIGHBOURS` được cập nhật theo NDCG@10 tốt nhất.
# - Lưu ý: test set không tham gia vào bước lựa chọn siêu tham số.

# Đánh giá NDCG@10 trên valid set với các giá trị neighbour-K khác nhau
# ITEM_K_LIST là candidate list cho số lân cận của ItemKNN.
# K_VALUES là cutoff list dùng trong evaluation metrics.
ITEM_K_LIST = [5, 10, 20, 50]

print("Tuning K_NEIGHBOURS trên Valid set (NDCG@10)...")
print(f"ITEM_K_LIST = {ITEM_K_LIST}")
print(f"{'K_neigh':>8} | {'NDCG@10':>10} | {'HR@10':>10}")
print("-" * 35)

best_k     = K_NEIGHBOURS
best_ndcg  = -1.0

for k_cand in ITEM_K_LIST:
    vm = evaluate_metrics_itemknn(
        eval_df=valid_df,
        train_df=train_df,
        R=R,
        R_T_norm=R_T_norm,
        user2idx=user2idx,
        movie2idx=movie2idx,
        idx2movie=idx2movie,
        k_values=(10,),
        k_neighbours=k_cand,
    )
    ndcg10 = vm[10]["ndcg"]
    hr10   = vm[10]["hr"]
    print(f"{k_cand:>8} | {ndcg10:>10.4f} | {hr10:>10.4f}")

    if ndcg10 > best_ndcg:
        best_ndcg = ndcg10
        best_k    = k_cand

print(f"\n[ItemKNN] Best K_NEIGHBOURS = {best_k}  (Valid NDCG@10 = {best_ndcg:.4f})")
K_NEIGHBOURS = best_k  # Cập nhật

Tuning K_NEIGHBOURS trên Valid set (NDCG@10)...
ITEM_K_LIST = [5, 10, 20, 50]
 K_neigh |    NDCG@10 |      HR@10
-----------------------------------
       5 |     0.0097 |     0.0167
      10 |     0.0095 |     0.0167
      20 |     0.0090 |     0.0167
      50 |     0.0089 |     0.0167

[ItemKNN] Best K_NEIGHBOURS = 5  (Valid NDCG@10 = 0.0097)


### 3.3 Đánh giá trên tập validation
Cell bên dưới báo cáo đầy đủ các chỉ số Top-$K$ trên tập validation sau khi đã chọn số láng giềng phù hợp.


In [14]:
# [Giải thích cell]
# - Mục đích: báo cáo metric trên tập validation sau khi mô hình/hyperparameter đã sẵn sàng.
# - Đầu vào chính: tập validation và các artifact cần thiết để sinh ranking.
# - Kết quả tạo ra: `valid_metrics`, dùng để kiểm tra hiệu năng trước khi đánh giá test.

print("[ItemKNN] Đánh giá trên VALID set...")
valid_metrics = evaluate_metrics_itemknn(
    eval_df=valid_df,
    train_df=train_df,
    R=R,
    R_T_norm=R_T_norm,
    user2idx=user2idx,
    movie2idx=movie2idx,
    idx2movie=idx2movie,
    k_values=K_VALUES,
    k_neighbours=K_NEIGHBOURS,
)
print_metrics(valid_metrics, "VALID")

[ItemKNN] Đánh giá trên VALID set...

=== VALID (n_users=240) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0025 |   0.0125 |   0.0084 |   0.0125 |   0.0071
  10 |   0.0017 |   0.0167 |   0.0097 |   0.0167 |   0.0075
  20 |   0.0013 |   0.0250 |   0.0118 |   0.0250 |   0.0081
  50 |   0.0015 |   0.0750 |   0.0215 |   0.0750 |   0.0096


### 3.4 Đánh giá cuối trên tập kiểm thử
Hiệu năng cuối được báo cáo trên test-full, test-warm và test-cold. Bảng 2 cho phép phân tích mức độ phụ thuộc của ItemKNN vào các item đã có lịch sử tương tác trong train.


In [15]:
# [Giải thích cell]
# - Mục đích: chạy đánh giá cuối cùng trên các split validation/test đã xác định.
# - Đầu vào chính: mô hình đã huấn luyện, train interactions và các tập held-out.
# - Kết quả tạo ra: bảng metric cho full, warm và cold split nếu có.
# - Lưu ý: không dùng kết quả test để chọn siêu tham số.

print("=" * 65)
print("FINAL EVALUATION — ItemKNN BASELINE")
print(f"Cấu hình: threshold={POSITIVE_THRESHOLD}, K_neighbours={K_NEIGHBOURS}")
print(f"Similarity: Cosine (binary implicit, no side information)")
print(f"Train items: {n_movies:,}  |  Train users: {n_users:,}")
print("=" * 65)

common_kwargs = dict(
    train_df=train_df,
    R=R,
    R_T_norm=R_T_norm,
    user2idx=user2idx,
    movie2idx=movie2idx,
    idx2movie=idx2movie,
    k_values=K_VALUES,
    k_neighbours=K_NEIGHBOURS,
)

for label, df in [
    ("VALID",     valid_df),
    ("TEST",      test_df),
    ("TEST_WARM", test_warm_df),
    ("TEST_COLD", test_cold_df),
]:
    print(f"\nĐánh giá {label}...")
    metrics = evaluate_metrics_itemknn(eval_df=df, **common_kwargs)
    print_metrics(metrics, label)

FINAL EVALUATION — ItemKNN BASELINE
Config: threshold=7, K_neighbours=5
Similarity: Cosine (binary implicit, no side information)
Train items: 75,115  |  Train users: 474

Đánh giá VALID...

=== VALID (n_users=240) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0025 |   0.0125 |   0.0084 |   0.0125 |   0.0071
  10 |   0.0017 |   0.0167 |   0.0097 |   0.0167 |   0.0075
  20 |   0.0013 |   0.0250 |   0.0118 |   0.0250 |   0.0081
  50 |   0.0015 |   0.0750 |   0.0215 |   0.0750 |   0.0096

Đánh giá TEST...

=== TEST (n_users=262) ===
   K |     Prec |   Recall |     NDCG |       HR |      MRR
----------------------------------------------------------
   5 |   0.0031 |   0.0153 |   0.0115 |   0.0153 |   0.0103
  10 |   0.0019 |   0.0191 |   0.0129 |   0.0191 |   0.0109
  20 |   0.0011 |   0.0229 |   0.0139 |   0.0229 |   0.0113
  50 |   0.0012 |   0.0611 |   0.0213 |   0.0611 |   0.0124

Đánh giá TEST_WA

### 3.5 Suy luận cho một người dùng
Cell này minh họa danh sách gợi ý top-$N$ cho một người dùng cụ thể. Các item đã xuất hiện trong train được loại bỏ để phản ánh đúng bối cảnh triển khai gợi ý thực tế.


In [16]:
# [Giải thích cell]
# - Mục đích: minh họa quy trình suy luận top-N cho một người dùng cụ thể.
# - Đầu vào chính: user_id, mô hình hoặc danh sách popularity, metadata phim và train history.
# - Kết quả tạo ra: bảng phim được đề xuất sau khi loại bỏ item người dùng đã thấy.
# - Lưu ý: đây là ví dụ định tính, không thay thế cho đánh giá định lượng Top-K.

def recommend_for_user_itemknn(
    user_id, R, R_T_norm, movies_df,
    user2idx, movie2idx, idx2movie,
    train_df, k_neighbours=K_NEIGHBOURS, n_recs=10,
):
    """Gợi ý top-N movies cho 1 user bằng itemKNN, loại trừ phim đã xem trong train."""
    if user_id not in user2idx:
        print(f"User {user_id} không có trong train set.")
        return pd.DataFrame()

    u_idx  = user2idx[user_id]
    scores = get_user_scores_itemknn(u_idx, R, R_T_norm, k_neighbours)

    # Mask items đã xem
    seen_ids = set(train_df[train_df["id"] == user_id]["movie_id"].values)
    for mid in seen_ids:
        if mid in movie2idx:
            scores[movie2idx[mid]] = -np.inf

    top_indices   = np.argsort(-scores)[:n_recs]
    top_movie_ids = [idx2movie[i] for i in top_indices]

    result = movies_df[movies_df["id"].isin(top_movie_ids)][
        ["id", "original_title", "genres", "year_published", "rate"]
    ].copy()
    result["itemknn_score"] = [scores[movie2idx[mid]] for mid in result["id"]]
    return result.sort_values("itemknn_score", ascending=False)


# Minh họa suy luận
sample_user = train_df["id"].iloc[0]
print(f"[ItemKNN] Gợi ý top-10 cho user_id = {sample_user}\n")
recs = recommend_for_user_itemknn(
    user_id=sample_user,
    R=R, R_T_norm=R_T_norm,
    movies_df=movies_df,
    user2idx=user2idx, movie2idx=movie2idx, idx2movie=idx2movie,
    train_df=train_df,
    k_neighbours=K_NEIGHBOURS,
    n_recs=10,
)
print(recs.to_string(index=False))

[ItemKNN] Gợi ý top-10 cho user_id = 103007

    id                           original_title                                          genres  year_published  rate  itemknn_score
445499 Sherlock: The Hounds of Baskerville (TV)                                         Intriga          2012.0   7.2       4.095905
173696                           Shutter Island                                Thriller|Intriga          2010.0   7.6       3.972810
800220                                  Aladdin Animación|Fantástico|Musical|Infantil|Aventuras          1992.0   7.4       3.887869
186830                                      300              Acción|Bélico|Aventuras|Fantástico          2006.0   7.2       3.796942
966887                     Beauty and the Beast   Animación|Fantástico|Romance|Musical|Infantil          1991.0   7.3       3.769878
779937                             Nightcrawler                                        Thriller          2014.0   7.3       3.763311
531662                  

## 4. Kết luận (Conclusion)
ItemKNN là baseline lọc cộng tác có khả năng khai thác cấu trúc đồng tương tác giữa các item, nhưng hiệu quả phụ thuộc trực tiếp vào độ phủ của ma trận train. Mô hình phù hợp để đánh giá sức mạnh của tín hiệu tương tác thuần túy và là mốc đối chiếu cho LightFM cũng như SeRel-LightFM khi các mô hình đó bổ sung metadata, văn bản và đồ thị tri thức.
